# Optional Notebook 5E — Downloading BIDS EEG from OpenNeuro, then inspecting ICA and an ERP

**Quantitative Methods in Neuroscience · Master in Neuroscience · University of Geneva**  
**Optional extension — not assessed**

This notebook demonstrates a second route to public neuroscience data. It uses `openneuro-py` to download **one run only** from the OpenNeuro dataset:

> **“EEG data from an auditory oddball task”** — OpenNeuro accession `ds003061`, snapshot `1.1.1`.

The workflow shows how to:

1. download selected files rather than a complete dataset;
2. inspect the Brain Imaging Data Structure (BIDS) folder and metadata tables;
3. open the EEGLAB recording with MNE-Python;
4. perform a minimal EEG preprocessing demonstration;
5. fit and plot a short ICA demonstration;
6. create standard and oddball epochs;
7. plot a simple auditory P300 comparison.

> This notebook emphasizes data organization, provenance, and reuse. It is not a complete P300 analysis.

## 0 · Before running

Use the dedicated **Python (qmn-optional)** environment created from the included `environment.yml`.

The selected `.set` recording is approximately 61 MB. The notebook stores it under:

```text
notebooks/data/optional_openneuro/ds003061/
```

Public OpenNeuro datasets can be downloaded without an account. The notebook pins the dataset snapshot so the classroom example remains reproducible.

> **Kernel requirement:** Run this notebook with **Python (qmn-optional)**. The first code cell stops early when the base Conda environment is selected. If a previous OpenNeuro download was interrupted, the loading cell removes only the selected run and downloads it once more.


In [ ]:
from pathlib import Path
import os
import shutil
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import mne
import openneuro as on

mne.set_log_level("WARNING")
RUN_DOWNLOAD = True
FORCE_REDOWNLOAD = False  # Change to True only when you want a completely fresh copy.

DATASET_ID = "ds003061"
SNAPSHOT = "1.1.1"
SUBJECT = "001"
RUN = "1"
TARGET_DIR = Path("data") / "optional_openneuro" / DATASET_ID

print("Python executable:", sys.executable)
print("Conda environment:", os.environ.get("CONDA_DEFAULT_ENV", "not reported"))
print("Python:", sys.version.split()[0])
print("MNE-Python:", mne.__version__)
print("SciPy:", scipy.__version__)
print("Target directory:", TARGET_DIR.resolve())

# A normal qmn kernel generally has qmn-optional as the environment directory name.
if Path(sys.prefix).name.lower() != "qmn-optional":
    raise RuntimeError(
        "This notebook is not running in the qmn-optional environment. "
        "In Jupyter, select Kernel -> Change Kernel -> Python (qmn-optional), "
        "restart the kernel, and run the notebook again."
    )


## 1 · Download only one participant and one run

`openneuro.download()` supports glob-style include patterns. The pattern below retrieves the files associated with subject `001`, task `P300`, run `1`, while OpenNeuro also supplies essential dataset-level BIDS metadata.

Downloading selected files is useful when the complete dataset is much larger than the classroom example requires.

In [ ]:
def remove_selected_run():
    """Remove only the selected subject/run so OpenNeuro must download it again."""
    eeg_dir = TARGET_DIR / f"sub-{SUBJECT}" / "eeg"
    if not eeg_dir.exists():
        return

    removed = []
    for path in eeg_dir.glob(f"*run-{RUN}*"):
        if path.is_file():
            path.unlink()
            removed.append(path.name)

    if removed:
        print("Removed cached run files:")
        for name in removed:
            print(" -", name)


def download_selected_run(force=False):
    if force:
        remove_selected_run()

    on.download(
        dataset=DATASET_ID,
        tag=SNAPSHOT,
        target_dir=TARGET_DIR,
        include=[f"sub-{SUBJECT}/eeg/*run-{RUN}*"],
    )
    print("Download complete or files already cached.")


if RUN_DOWNLOAD:
    download_selected_run(force=FORCE_REDOWNLOAD)
else:
    print("Download skipped in offline-validation mode.")


## 2 · Inspect the BIDS file tree

BIDS stores the signal together with machine-readable metadata. For this run, important sidecar files include:

- `_eeg.set`: the EEG signal in EEGLAB format;
- `_eeg.json`: acquisition metadata;
- `_channels.tsv`: channel types and status;
- `_events.tsv`: event onsets and trial descriptions;
- `_electrodes.tsv` and `_coordsystem.json`: sensor locations and coordinate metadata.

In [ ]:
if RUN_DOWNLOAD:
    downloaded_files = sorted(
        path.relative_to(TARGET_DIR)
        for path in TARGET_DIR.rglob("*")
        if path.is_file()
    )
    for path in downloaded_files:
        print(path)

## 3 · Read the BIDS tables

The tab-separated metadata files can be inspected directly with pandas before the electrophysiology signal is loaded.

In [ ]:
if RUN_DOWNLOAD:
    eeg_dir = TARGET_DIR / f"sub-{SUBJECT}" / "eeg"
    stem = f"sub-{SUBJECT}_task-P300_run-{RUN}"

    set_path = eeg_dir / f"{stem}_eeg.set"
    channels_path = eeg_dir / f"{stem}_channels.tsv"
    events_path = eeg_dir / f"{stem}_events.tsv"

    required = [set_path, channels_path, events_path]
    missing = [path for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError(f"Expected OpenNeuro files were not found: {missing}")

    set_size_mb = set_path.stat().st_size / 1024**2
    print(f"Signal file: {set_path}")
    print(f"Signal file size: {set_size_mb:.1f} MB")

    # The selected recording is around 60 MB. A much smaller file is incomplete.
    if set_size_mb < 50:
        print("The .set file is too small and is probably incomplete. Re-downloading it now.")
        download_selected_run(force=True)
        set_size_mb = set_path.stat().st_size / 1024**2
        print(f"New signal file size: {set_size_mb:.1f} MB")

    channels_table = pd.read_csv(channels_path, sep="\t")
    events_table = pd.read_csv(events_path, sep="\t")

    display(channels_table.head())
    display(events_table.head())
    display(events_table["trial_type"].value_counts().head(10))


## 4 · Open the EEGLAB recording with MNE

The `.set` file in this snapshot contains the recording data, so it can be read directly with `mne.io.read_raw_eeglab()`.

In [ ]:
if RUN_DOWNLOAD:
    try:
        raw = mne.io.read_raw_eeglab(set_path, preload=True, verbose=False)
    except (OSError, EOFError, ValueError) as first_error:
        print("MNE could not read the cached .set file:")
        print(repr(first_error))
        print("Deleting the selected run and performing one clean re-download...")

        download_selected_run(force=True)

        try:
            raw = mne.io.read_raw_eeglab(set_path, preload=True, verbose=False)
        except Exception as second_error:
            raise RuntimeError(
                "The file was still unreadable after a clean re-download. "
                "Confirm that the notebook uses the Python (qmn-optional) kernel and rerun "
                "the environment update from environment.yml."
            ) from second_error

    # Keep EEG channels for this short demonstration.
    raw.pick("eeg")

    # Attach a standard montage as a best-effort teaching visualization.
    # The BIDS electrodes and coordinate files remain the authoritative metadata.
    try:
        raw.set_montage("biosemi64", match_case=False, on_missing="ignore")
    except ValueError:
        raw.set_montage("standard_1020", match_case=False, on_missing="ignore")

    summary = {
        "duration_minutes": raw.times[-1] / 60,
        "sampling_frequency_hz": raw.info["sfreq"],
        "n_eeg_channels": raw.info["nchan"],
        "n_annotations": len(raw.annotations),
    }
    display(summary)


## 5 · Plot a short raw segment

Before preprocessing, inspect whether the signal contains large transients, drifting channels, or movement-related contamination.

In [ ]:
if RUN_DOWNLOAD:
    raw.plot(
        start=0,
        duration=10,
        n_channels=20,
        scalings="auto",
        show_scrollbars=False,
        show=False,
        title="OpenNeuro ds003061 — raw EEG preview",
    )
    plt.show()

## 6 · Minimal preprocessing for an ERP demonstration

For this brief example we:

- apply an average EEG reference;
- filter from 0.5 to 30 Hz;
- keep the annotations imported with the EEGLAB file.

A real analysis should also define bad-channel detection, artifact correction, rejection thresholds, and participant-level quality control.

In [ ]:
if RUN_DOWNLOAD:
    raw_preprocessed = raw.copy()
    raw_preprocessed.set_eeg_reference("average", projection=False, verbose=False)
    raw_preprocessed.filter(l_freq=0.5, h_freq=30.0, verbose=False)

    events, event_id = mne.events_from_annotations(raw_preprocessed, verbose=False)
    print("Annotation labels imported by MNE:")
    display(pd.Series(event_id, name="event_code"))

## 7 · Fit and plot ICA on a short OpenNeuro segment

OpenNeuro provides the data; MNE performs the signal processing. To connect the two ideas, this section fits a small ICA model to the first two minutes of the downloaded recording.

As in Notebook 5C, the fitting copy is high-pass filtered at 1 Hz. The purpose is only to visualize component topographies and source time courses. No component is removed automatically.

In [ ]:
if RUN_DOWNLOAD:
    raw_for_ica = raw.copy()
    raw_for_ica.filter(l_freq=1.0, h_freq=40.0, verbose=False)
    raw_for_ica.set_eeg_reference("average", projection=False, verbose=False)

    # Limit the classroom example to at most two minutes for speed.
    raw_for_ica.crop(tmax=min(120.0, raw_for_ica.times[-1]))

    openneuro_ica = mne.preprocessing.ICA(
        n_components=15,
        method="fastica",
        random_state=97,
        max_iter="auto",
    )
    openneuro_ica.fit(raw_for_ica, decim=2, verbose=False)
    print(openneuro_ica)

In [ ]:
if RUN_DOWNLOAD:
    openneuro_ica.plot_components(
        picks=range(15),
        inst=raw_for_ica,
        show_names=True,
        show=False,
    )
    plt.show()

    openneuro_ica.plot_sources(
        raw_for_ica,
        start=0,
        stop=min(20, raw_for_ica.times[-1]),
        show=False,
        title="OpenNeuro ICA source time courses — inspect before excluding",
    )
    plt.show()

    COMPONENTS_TO_REMOVE = []
    print("Components selected for removal:", COMPONENTS_TO_REMOVE)

## 8 · Find standard and oddball events

The dataset uses the label `stimulus/standard` and an oddball label containing `oddball`. The code searches the labels rather than assuming fixed integer codes.

In [ ]:
if RUN_DOWNLOAD:
    standard_key = next(
        key for key in event_id
        if "stimulus/standard" in key.lower()
    )
    oddball_key = next(
        key for key in event_id
        if "oddball" in key.lower()
    )

    selected_event_id = {
        "standard": event_id[standard_key],
        "oddball": event_id[oddball_key],
    }
    print("Selected events:", selected_event_id)

## 9 · Epoch the EEG around each stimulus

Epochs run from 100 ms before to 600 ms after stimulus onset. The pre-stimulus interval is used as the baseline.

In [ ]:
if RUN_DOWNLOAD:
    epochs = mne.Epochs(
        raw_preprocessed,
        events,
        event_id=selected_event_id,
        tmin=-0.1,
        tmax=0.6,
        baseline=(-0.1, 0.0),
        preload=True,
        reject_by_annotation=True,
        verbose=False,
    )

    print(epochs)
    print("Standards:", len(epochs["standard"]))
    print("Oddballs:", len(epochs["oddball"]))

## 10 · Plot the standard and oddball responses at Cz

The P300 is commonly inspected at central-parietal electrodes. This single-run figure is descriptive and should not be treated as a population-level result.

In [ ]:
if RUN_DOWNLOAD:
    evokeds = {
        "Standard": epochs["standard"].average(),
        "Oddball": epochs["oddball"].average(),
    }

    channel_lookup = {name.lower(): name for name in epochs.ch_names}
    cz = channel_lookup.get("cz", epochs.ch_names[0])
    print("Displayed channel:", cz)

    mne.viz.plot_compare_evokeds(
        evokeds,
        picks=[cz],
        combine="mean",
        show=False,
        title=f"Auditory oddball ERP at {cz}",
    )
    plt.show()

## 11 · Plot an oddball-minus-standard topography

The difference wave highlights sensor-level differences between the two average responses. We display the mean difference from 250 to 400 ms, a conventional descriptive P300 window.

In [ ]:
if RUN_DOWNLOAD:
    difference = mne.combine_evoked(
        [evokeds["Oddball"], evokeds["Standard"]],
        weights=[1, -1],
    )
    difference.comment = "Oddball minus standard"

    difference.plot_topomap(
        times=[0.325],
        average=0.150,
        ch_type="eeg",
        contours=6,
        show=False,
        time_unit="s",
    )
    plt.show()

## What students should remember

- OpenNeuro distributes versioned, BIDS-organized public datasets.
- `openneuro-py` can retrieve selected participants or files instead of a complete dataset.
- BIDS separates signal files from structured metadata tables and JSON sidecars.
- MNE can read several BIDS-compatible EEG formats, including EEGLAB `.set` files.
- A public repository and a preprocessing package serve different roles: OpenNeuro distributes the versioned data, while MNE processes and visualizes the signal.
- ICA components must be inspected before any exclusion; this notebook removes none by default.
- Downloading public data does not remove the need to cite the dataset and document its exact snapshot.
- This one-run ERP is a software demonstration, not a confirmatory result.

## References and documentation

- MNE-Python documentation: [Repairing artifacts with ICA](https://mne.tools/stable/auto_tutorials/preprocessing/40_artifact_correction_ica.html).
- OpenNeuro documentation: [User Guide](https://docs.openneuro.org/user_guide.html) and [API/advanced access](https://docs.openneuro.org/api.html).
- `openneuro-py` documentation: [Python client for accessing OpenNeuro datasets](https://pypi.org/project/openneuro-py/).
- Delorme, A. (2022). **“EEG data from an auditory oddball task.”** OpenNeuro dataset `ds003061`, snapshot `1.1.1`, DOI: `10.18112/openneuro.ds003061.v1.1.1`.